In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import shutil

# project specific imports
from annotation_methods.vlm_helpers import make_zero_shot_predictions, benchmark_batch_size, make_zero_shot_predictions_minimal

/home/warredv/miniconda3/envs/main-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_ROOT = Path("../../Data")
RESULTS_EXP_1_PATH = Path("../../Results/Experiment_1")
RESULTS_EXP_2_1_PATH = Path("../../Results/Experiment_2.1")
RESULTS_EXP_2_2_PATH = Path("../../Results/Experiment_2.2")

DATASET_DICT = {
    "apples": ["good apple", "bad apple"],
    "tomatoes": ["tomato"],
}

models_dict = {
    "gd_t": {
        "is_base_model": False,
        "name": "grounding dino tiny",
        "max_batch_size": 4,
    },
    "gd_b": {
        "is_base_model": True,
        "name": "grounding dino base",
        "max_batch_size": 4,
    },
    "owlvit_b_16": {
        "is_base_model": True,
        "name": "owl VIT base patch 16",
        "max_batch_size": 16,
    },
    # "owlvit_b_32": {
    #     "is_base_model": True,
    #     "name": "owl VIT base patch 32",
    #     "max_batch_size": 32,
    # },
    "owlvit_l_14": {
        "is_base_model": False,
        "name": "owl VIT large patch 14",
        "max_batch_size": 4,
    },    
    "mmgd_t": {
        "is_base_model": False,
        "name": "MM-Grounding Dino tiny",
        "max_batch_size": 4,
    },
    "mmgd_b_all": {
        "is_base_model": True,
        "name": "MM-Grounding Dino base",
        "max_batch_size": 4,
    },
    "mmgd_l_all": {
        "is_base_model": False,
        "name": "MM-Grounding Dino large",
        "max_batch_size": 1,
    },
}


## Batch size experiment

In [ ]:
all_dfs = []

for model_key, cfg in models_dict.items():
    print(f"\n=== Benchmarking {model_key} ({cfg['name']}) ===")

    for bs in [1, 2, 4, 8, 16, 32]:
        metrics, df = benchmark_batch_size(
            images_folder=str(DATA_ROOT / "apples" / "images" / "train"),
            categories_list=["good apple","bad apple"],
            model_name=model_key,
            batch_size=bs,
            sample_size=32,
            warmup_steps=3,
        )

        all_dfs.append(df)

        if metrics["oom"]:
            print(f"Stopping {model_key} because batch_size={bs} caused OOM")
            break

benchmarks_df = pd.concat(all_dfs, ignore_index=True)


=== Benchmarking gd_t (grounding dino tiny) ===
torch.cuda.is_available(): True
Found 121 image files
[batch=1] images=32 | time=5.13s | throughput=6.24 img/s | peak_mem=1867.5 MB | oom=False
torch.cuda.is_available(): True
Found 121 image files
[batch=2] images=32 | time=4.73s | throughput=6.76 img/s | peak_mem=3050.2 MB | oom=False
torch.cuda.is_available(): True
Found 121 image files
[batch=4] images=32 | time=4.59s | throughput=6.96 img/s | peak_mem=5401.1 MB | oom=False
torch.cuda.is_available(): True
Found 121 image files
[batch=8] images=32 | time=4.54s | throughput=7.04 img/s | peak_mem=10092.4 MB | oom=False
torch.cuda.is_available(): True
Found 121 image files
CUDA OOM at batch_size=16 (processed 0 images so far)
[batch=16] images=0 | time=0.00s | throughput=0.00 img/s | peak_mem=18245.7 MB | oom=True
Stopping gd_t because batch_size=16 caused OOM

=== Benchmarking gd_b (grounding dino base) ===
torch.cuda.is_available(): True
Found 121 image files
[batch=1] images=32 | time

In [5]:
benchmarks_df["total_time_s"] = benchmarks_df["total_time_s"].round(2)
benchmarks_df["images_per_second"] = benchmarks_df["images_per_second"].round(2)
benchmarks_df["peak_mem_mb"] = benchmarks_df["peak_mem_mb"].round(0).astype("int64")
benchmarks_df

,model,batch_size,num_images,total_time_s,images_per_second,peak_mem_mb,oom
0,gd_t,1,32,5.13,6.24,1868,False
1,gd_t,2,32,4.73,6.76,3050,False
2,gd_t,4,32,4.59,6.96,5401,False
3,gd_t,8,32,4.54,7.04,10092,False
4,gd_t,16,0,0.00,0.00,18246,True
5,gd_b,1,32,6.99,4.58,2114,False
6,gd_b,2,32,6.47,4.94,3308,False
7,gd_b,4,32,6.32,5.07,5667,False
8,gd_b,8,32,6.22,5.14,10382,False
9,gd_b,16,16,3.10,5.16,19058,True


In [6]:
all_dfs = []

for model_key, cfg in models_dict.items():
    print(f"\n=== Benchmarking {model_key} ({cfg['name']}) ===")

    for bs in [1, 2, 4, 8, 16, 32]:
        metrics, df = benchmark_batch_size(
            images_folder=str(DATA_ROOT / "tomatoes" / "images" / "train"),
            categories_list=["tomato"],
            model_name=model_key,
            batch_size=bs,
            random_state=42,
            sample_size=128,
            warmup_steps=3,
        )

        all_dfs.append(df)

        if metrics["oom"]:
            print(f"Stopping {model_key} because batch_size={bs} caused OOM")
            break

benchmarks_df = pd.concat(all_dfs, ignore_index=True)



=== Benchmarking gd_t (grounding dino tiny) ===
torch.cuda.is_available(): True
Found 5961 image files
[batch=1] images=128 | time=22.33s | throughput=5.73 img/s | peak_mem=5710.5 MB | oom=False
torch.cuda.is_available(): True
Found 5961 image files
[batch=2] images=128 | time=23.86s | throughput=5.37 img/s | peak_mem=9214.6 MB | oom=False
torch.cuda.is_available(): True
Found 5961 image files
[batch=4] images=128 | time=25.22s | throughput=5.08 img/s | peak_mem=14366.5 MB | oom=False
torch.cuda.is_available(): True
Found 5961 image files
CUDA OOM at batch_size=8 (processed 88 images so far)
[batch=8] images=88 | time=17.02s | throughput=5.17 img/s | peak_mem=19128.3 MB | oom=True
Stopping gd_t because batch_size=8 caused OOM

=== Benchmarking gd_b (grounding dino base) ===
torch.cuda.is_available(): True
Found 5961 image files
[batch=1] images=128 | time=30.12s | throughput=4.25 img/s | peak_mem=5957.7 MB | oom=False
torch.cuda.is_available(): True
Found 5961 image files
[batch=2] im

In [7]:
benchmarks_df["total_time_s"] = benchmarks_df["total_time_s"].round(2)
benchmarks_df["images_per_second"] = benchmarks_df["images_per_second"].round(2)
benchmarks_df["peak_mem_mb"] = benchmarks_df["peak_mem_mb"].round(0).astype("int64")
benchmarks_df

,model,batch_size,num_images,total_time_s,images_per_second,peak_mem_mb,oom
0,gd_t,1,128,22.33,5.73,5711,False
1,gd_t,2,128,23.86,5.37,9215,False
2,gd_t,4,128,25.22,5.08,14366,False
3,gd_t,8,88,17.02,5.17,19128,True
4,gd_b,1,128,30.12,4.25,5958,False
5,gd_b,2,128,32.53,3.94,9483,False
6,gd_b,4,128,34.27,3.73,14659,False
7,gd_b,8,88,23.12,3.81,19448,True
8,owlvit_b_16,1,128,8.72,14.68,2965,False
9,owlvit_b_16,2,128,8.30,15.42,2441,False


## Experiment 1

the categories list order has to be the same as the gt order

In [3]:
# Run tests on the base models
for dataset in DATASET_DICT.keys():
    for model_name, model_cfg in models_dict.items():
        if model_cfg["is_base_model"]:
            make_zero_shot_predictions(
                images_folder=str(DATA_ROOT / dataset / "images" / "test"),
                categories_list=DATASET_DICT.get(dataset),
                model_name=model_name,
                batch_size=model_cfg["max_batch_size"],
                # sample_size=10,
                output_path=str(RESULTS_EXP_1_PATH),
            )

Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_gd_b_predictions.json
Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_owlvit_b_16_predictions.json
Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_mmgd_b_all_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_gd_b_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_owlvit_b_16_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_mmgd_b_all_predictions.json


In [3]:
make_zero_shot_predictions(
    images_folder= str(DATA_ROOT / "tomatoes" / "images" / "test"),
    categories_list=["tomato"],
    model_name="owlvit_b_32",
    batch_size=4,
    sample_size=16,
    output_path= str(RESULTS_EXP_1_PATH),
    # threshold=0.25,
    # text_threshold=0.25,
)

Found 662 image files
[postprocess] using threshold = 0.1
[postprocess] using threshold = 0.1
[postprocess] using threshold = 0.1
[postprocess] using threshold = 0.1
Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_owlvit_b_32_predictions.json


## Experiment 2.1

In [4]:
# copy the results that are already made
for dataset in DATASET_DICT.keys():
    for model_name, model_cfg in models_dict.items():
        if model_cfg["is_base_model"]:

            filename = f"{dataset}_test_{model_name}_predictions.json"
            src = RESULTS_EXP_1_PATH / filename
            dst = RESULTS_EXP_2_1_PATH / filename

            if src.exists():
                RESULTS_EXP_2_1_PATH.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
            else:
                print(f"Missing file: {src}")


In [5]:
# Run additional tests on the non base models
for dataset in DATASET_DICT.keys():
    for model_name, model_cfg in models_dict.items():
        if not model_cfg["is_base_model"]:
            make_zero_shot_predictions(
                images_folder=str(DATA_ROOT / dataset / "images" / "test"),
                categories_list=DATASET_DICT.get(dataset),
                model_name=model_name,
                batch_size=model_cfg["max_batch_size"],
                # sample_size=10,
                output_path=str(RESULTS_EXP_2_1_PATH),
            )

Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/apples_test_gd_t_predictions.json
Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/apples_test_owlvit_l_14_predictions.json
Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/apples_test_mmgd_t_predictions.json
Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/apples_test_mmgd_l_all_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/tomatoes_test_gd_t_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/tomatoes_test_owlvit_l_14_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/tomatoes_test_mmgd_t_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_2.1/tomatoes_test_mmgd_l_all_predictions.json


## Experiment 2.2

In [ ]:
make_zero_shot_predictions_minimal(
    images_folder= str(DATA_ROOT / "tomatoes" / "images" / "test"),
    categories_list=["tomato"],
    model_name="mmgd_t",
    batch_size=8,
    sample_size=16,
    # threshold=0.25,
    # text_threshold=0.25,
)

post_process_grounded_object_detection defaults:
  outputs = <required>
  input_ids = None
  threshold = 0.25
  text_threshold = 0.25
  target_sizes = None
  text_labels = None
Found 662 image files
Original sizes: [(124, 229), (167, 249), (141, 296), (154, 221), (115, 180), (141, 286), (124, 134), (152, 156)]
Processed tensor shape: (8, 3, 1333, 800)


: 